# Thursday — Making Your Model Small (Export · Quantize · Benchmark)

On **Wednesday** you trained a recognizer and saved two files: `recognizer.pt` (the weights) and `recognizer.onnx` (the portable version). Today you make that model **small and fast enough to run on a Raspberry Pi** — and you *measure* the difference instead of guessing.

The plan follows the three numbers from the lecture — **size, speed, accuracy**:
1. **Export** cleanly to ONNX and prove the export didn't change your model.
2. **Measure** the original (FP32): how big, how fast, how accurate.
3. **Quantize** it to INT8 (8-bit numbers instead of 32-bit).
4. **Measure again** and build the FP32-vs-INT8 comparison table — proof of the trade-off you made.
5. Get everything **export-ready** to deploy to the robot arm on Friday.

This notebook has several **🔧 YOUR TURN** cells — spots where *you* fill in the missing line. Look for `# TODO` and the blanks written as `____`.


> **How to run this:** You're on our GPU server through **JupyterHub**, in your browser — everything is installed. Run each cell with **Shift+Enter**, top to bottom. Today we deliberately run the models on the **CPU**, because that's the closest thing on this server to the Pi you'll deploy to on Friday. (Your real Pi numbers come tomorrow.)

In [ ]:
# You're on our JupyterHub GPU server — these are already installed.
# Only if a later cell complains something is missing:
# !pip install onnx onnxruntime torch torchvision pillow matplotlib


In [ ]:
import os, time
import numpy as np
import onnxruntime as ort
import onnx
print("onnxruntime", ort.__version__)

def single_file(path):
    """Keep an .onnx as ONE self-contained file (embed weights, drop any .onnx.data)."""
    m = onnx.load(path)                    # pulls in external weights if present
    onnx.save_model(m, path, save_as_external_data=False)
    if os.path.exists(path + ".data"):
        os.remove(path + ".data")         # the sidecar is no longer needed


## 1. Load your Wednesday model

We rebuild the MobileNetV2 shape, then load **your** trained weights from `recognizer.pt`. We also grab your **class names** (their order matters — the arm reads predictions by position) and set up the **validation set** so we can check accuracy later.

*Wednesday's files live in the `wednesday/` folder next to this one (`WED_DIR` below). If they're missing, re-pull Wednesday or re-run its notebook first.*


In [ ]:
import torch, torch.nn as nn
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2

# Load YOUR trained weights + class order saved on Wednesday.
# nbgitpuller gives each day its own folder, so Wednesday's files are one level up:
WED_DIR = "../wednesday"   # change this if your Wednesday files live elsewhere
ckpt = torch.load(os.path.join(WED_DIR, "recognizer.pt"), map_location="cpu")
CLASS_NAMES = ckpt["classes"]
NUM_CLASSES = len(CLASS_NAMES)
print("Your classes:", CLASS_NAMES)

# Rebuild the architecture (no download), then load your weights into it.
model = mobilenet_v2(weights=None)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
model.load_state_dict(ckpt["state_dict"])
model.eval()
print("Loaded your recognizer.")


In [ ]:
# Same preprocessing you used on Wednesday (so the model sees images the way it was trained).
IMG_SIZE = 224
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

DATA_DIR = os.path.join(WED_DIR, "my_objects")   # train/ and val/ from Wednesday
val_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), val_transform)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=16, shuffle=False)

assert val_ds.classes == CLASS_NAMES, "Class order doesn't match the saved model!"
print(f"{len(val_ds)} validation images across {NUM_CLASSES} classes")


## 2. Export to ONNX (the portable format)

ONNX is the universal format from the lecture: train in PyTorch, save as ONNX, and the **same file** runs on the Pi, a phone, a browser, or the car. We feed the exporter one **dummy image** so it can trace the shapes.

*(We pass `dynamo=False` to use PyTorch's classic ONNX exporter — on newer PyTorch the default exporter can choke when down-converting opsets; the classic one handles `dynamic_axes` and the opset cleanly. We then call `single_file()` so the model is one self-contained `.onnx` instead of a `.onnx` + `.onnx.data` pair — easier to measure and to copy to the arm.)*


In [ ]:
FP32_PATH = "recognizer.onnx"
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)   # one fake 224x224 color image

torch.onnx.export(
    model, dummy, FP32_PATH,
    input_names=["image"], output_names=["scores"],
    dynamic_axes={"image": {0: "batch"}, "scores": {0: "batch"}},  # allow any batch size
    opset_version=17,
    dynamo=False,   # use the classic exporter so dynamic_axes + opset behave predictably
)
single_file(FP32_PATH)               # fold weights back in -> one recognizer.onnx
print("Exported", FP32_PATH, "(", round(os.path.getsize(FP32_PATH)/1e6, 1), "MB )")


## 3. 🔧 YOUR TURN — did the export change anything?

An export can silently go wrong. The safe habit: run the **same image** through the original PyTorch model *and* the ONNX model and confirm the outputs match. If they don't, the ONNX file is bad and Friday would fail mysteriously — better to catch it now.


In [ ]:
# Start an ONNX Runtime session (CPU — our Pi stand-in).
sess_fp32 = ort.InferenceSession(FP32_PATH, providers=["CPUExecutionProvider"])

# Same input through both models:
x = dummy.numpy().astype(np.float32)
with torch.no_grad():
    torch_out = model(dummy).numpy()
onnx_out = sess_fp32.run(None, {"image": x})[0]

# TODO 1: the two outputs should be almost identical.
# Compute the LARGEST absolute difference between torch_out and onnx_out.
# hint: np.abs(a - b).max()
max_diff = ____

print("largest difference:", max_diff)
print("export looks:", "GOOD ✅" if max_diff < 1e-3 else "WRONG ❌ — re-export!")


## 4. The three measurements

We'll write one small helper for each number — **size**, **speed**, **accuracy** — then run them on your original model to get a baseline. You'll reuse the *exact same* helpers on the quantized model in Section 6, so the comparison is fair.


In [ ]:
def onnx_size_mb(path):
    # TODO 2: return the file size of `path` in MEGABYTES.
    # hint: os.path.getsize(path) gives bytes; 1 MB = 1_000_000 bytes.
    return ____


def benchmark_speed(session, n_runs=50):
    """Average time to classify one image, and frames per second."""
    one_image = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
    session.run(None, {"image": one_image})          # warm-up (NOT timed)
    start = time.time()
    for _ in range(n_runs):
        session.run(None, {"image": one_image})
    elapsed = time.time() - start                     # seconds for all n_runs
    # TODO 3: turn `elapsed` into the two numbers we report.
    avg_ms = ____     # average MILLISECONDS per image   (hint: seconds -> ms is x1000)
    fps    = ____     # images per second                (hint: how many runs per second?)
    return avg_ms, fps


def evaluate_accuracy(session):
    """Run the ONNX model over the whole validation set; return accuracy 0..1."""
    correct = total = 0
    for imgs, labels in val_loader:
        scores = session.run(None, {"image": imgs.numpy().astype(np.float32)})[0]
        preds = scores.argmax(axis=1)
        # TODO 5: count how many predictions matched the true labels.
        # hint: (preds == labels.numpy()).sum()
        correct += ____
        total   += len(labels)
    return correct / total


In [ ]:
# Baseline: measure the original FP32 model.
size_fp32 = onnx_size_mb(FP32_PATH)
ms_fp32, fps_fp32 = benchmark_speed(sess_fp32)
acc_fp32 = evaluate_accuracy(sess_fp32)
print(f"FP32  size {size_fp32:5.1f} MB   {ms_fp32:6.1f} ms/img   {fps_fp32:5.1f} FPS   accuracy {acc_fp32:.1%}")


## 5. 🔧 YOUR TURN — quantize to INT8

Quantization stores each weight with **fewer bits**: 32-bit floats become 8-bit integers — like rounding 3.14159265 to 3.14. The file gets ~4× smaller and integer math is faster, usually costing only ~1% accuracy.

`quantize_dynamic` does this for us. It takes the **input** ONNX path, an **output** path, and the target type. *(A warning about “preprocessing” or opset is normal — you can ignore it for our model.)*


In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

INT8_PATH = "recognizer_int8.onnx"

# TODO 4: quantize the FP32 model into INT8.
# Fill the three blanks: the input path, the output path, and weight_type=QuantType.QUInt8
quantize_dynamic(____, ____, weight_type=____)

print("Quantized ->", INT8_PATH)


## 6. Measure the quantized model

Same three helpers, new model. Nothing to fill in here — just run it and watch the numbers move.


In [ ]:
single_file(INT8_PATH)   # same one-file treatment so the size comparison is fair
sess_int8 = ort.InferenceSession(INT8_PATH, providers=["CPUExecutionProvider"])
size_int8 = onnx_size_mb(INT8_PATH)
ms_int8, fps_int8 = benchmark_speed(sess_int8)
acc_int8 = evaluate_accuracy(sess_int8)
print(f"INT8  size {size_int8:5.1f} MB   {ms_int8:6.1f} ms/img   {fps_int8:5.1f} FPS   accuracy {acc_int8:.1%}")


## 7. 🔧 YOUR TURN — the comparison table

Put the two models side by side. This is the evidence behind the edge-AI trade-off from the lecture: you traded a little accuracy for a big win in size and speed — and it's a great thing to show off on Friday.


In [ ]:
# TODO 6: fill in the INT8 row using YOUR int8 measurements (size_int8, fps_int8, acc_int8).
rows = [
    ("FP32 (original)",  size_fp32, fps_fp32, acc_fp32),
    ("INT8 (quantized)", ____,      ____,     ____),
]

print(f"{'model':18s}{'size (MB)':>11s}{'FPS':>8s}{'accuracy':>12s}")
print("-" * 49)
for name, size, fps, acc in rows:
    print(f"{name:18s}{size:11.1f}{fps:8.1f}{acc:12.1%}")

print(f"\nQuantizing made it {size_fp32/size_int8:.1f}x smaller and {fps_int8/fps_fp32:.1f}x faster,")
print(f"for a {(acc_fp32 - acc_int8)*100:.1f} percentage-point change in accuracy.")


### (optional) Draw it
A picture of the trade-off, like the slide. Run it as-is.


In [ ]:
import matplotlib.pyplot as plt
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 3))
a1.bar(["FP32", "INT8"], [size_fp32, size_int8], color=["#64748B", "#0891B2"])
a1.set_title("Size (MB) — smaller is better")
a2.bar(["FP32", "INT8"], [fps_fp32, fps_int8], color=["#64748B", "#22D3EE"])
a2.set_title("Speed (FPS) — higher is better")
plt.tight_layout(); plt.show()


## 8. 🔧 YOUR TURN — which one would you ship, and why?

There's no single right answer — that's the whole point of the trade-off. In the cell below, write **2–3 sentences**: which model you'd put on the arm for Friday, and what number convinced you.

> **Reminder:** these numbers came from this **server's CPU**, which is a *stand-in* for the Pi, not the Pi itself. The ranking (INT8 smaller/faster) will hold on real hardware; the exact FPS will differ. You'll re-measure on the actual Pi tomorrow.


*(your answer here)*

- I would deploy: 
- Because: 


## 9. Get ready for the arm

Everyone deploys to the **vision-guided robot arm** on Friday — and you've already done the hard part. The arm's camera crops each object and hands the crop to your model; your model names it; the arm picks whichever crop matches the **target** you name at run time.

- **Nothing new to code today** — Sections 3–6 already proved your ONNX model loads and predicts correctly.
- Tomorrow you'll upload `recognizer_int8.onnx` (or `recognizer.onnx`) to the arm's Raspberry Pi.
- The vision-only “recognize & label” mode (camera → label, no arm movement) is your **backup** if an arm misbehaves.

The helper below is a handy way to test your model on any saved image before tomorrow.


In [ ]:
# A handy tool: classify one saved image with your (quantized) model.
from PIL import Image
import numpy as np

def classify_image(path, session):
    img = Image.open(path).convert("RGB")
    x = val_transform(img).unsqueeze(0).numpy().astype(np.float32)
    scores = session.run(None, {"image": x})[0][0]
    e = np.exp(scores - scores.max()); probs = e / e.sum()   # softmax -> confidence
    idx = int(probs.argmax())
    return CLASS_NAMES[idx], float(probs[idx])

# Example (point it at any image file):
# label, conf = classify_image("test.jpg", sess_int8)
# print(f"I think this is a {label} ({conf:.0%} sure)")

# Try it: run classify_image() on a few of your own photos and check it's right.
# (This is exactly the call the arm makes on each crop tomorrow.)


## 10. Export-ready checklist — do this before you leave

Friday is hardware day; nobody should arrive without a deployable model. Confirm all four:

- [ ] `recognizer.onnx` exists and matched PyTorch (Section 3 said GOOD).
- [ ] `recognizer_int8.onnx` exists and you have its size/speed/accuracy in the table.
- [ ] Your **class names, in order**, are written down somewhere you'll have tomorrow: run the next cell and copy the output.
- [ ] You can name your **target** object for the arm (any of your classes).


In [ ]:
checks = [
    ("recognizer.pt (Wed)",  os.path.join(WED_DIR, "recognizer.pt")),
    ("recognizer.onnx",      "recognizer.onnx"),
    ("recognizer_int8.onnx", "recognizer_int8.onnx"),
]
print("Your files:")
for label, path in checks:
    ok = os.path.exists(path)
    mark = "✅" if ok else "❌ MISSING"
    size = f"{os.path.getsize(path)/1e6:.1f} MB" if ok else ""
    print(f"  {mark}  {label:22s} {size}")
print("\nWRITE THIS DOWN — your class order for Friday:")
print(" ", CLASS_NAMES)


## Challenges (if you finish early)

1. **QInt8 vs QUInt8.** Re-quantize with `weight_type=QuantType.QInt8` and compare. Smaller? Faster? More accurate on *your* objects?
2. **Static (calibration) quantization.** Dynamic quant is the easy path; *static* quant uses a few real images to calibrate and is usually more accurate. Try `quantize_static` with a small `CalibrationDataReader` over your val images.
3. **Per-class accuracy after quantizing.** Which object lost the most accuracy when you shrank the model? (Build a per-class count like Wednesday's Section 5.)
4. **Shrink the input.** Re-export at 160×160 instead of 224×224, then benchmark. How much faster? What did accuracy cost?
5. **Batch it.** Benchmark batch size 1 vs 8. Does throughput (images/second) change on CPU?
6. **Is ONNX Runtime actually faster?** Time the plain PyTorch model on CPU and compare to `sess_fp32`. Which wins, and by how much?
